# 017 diagnose: why the refiner barely trains

# Imports

In [ ]:
import os
import sys

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

from uavloc.segmentation.eval import HOLDOUT_LOCATION, STRIP_LOCATION, STRIP_ROWS, load_baseline_model
from uavloc.segmentation.losses import ce_dice_loss
from uavloc.segmentation.refine.refinement_cnn import (RefinementCNN, RefinerConfig, bank_batch, build_crop_bank,
                                                       cross_entropy, luma)
from uavloc.segmentation.refine.residual_refiner import ResidualRefiner, ResidualRefinerConfig
from uavloc.track import REPO_ROOT, set_seed

sys.path.insert(0, str(REPO_ROOT / "pocs" / "village-seg-baseline"))
import village_seg as vs

# Params

In [ ]:
CHECKPOINT = "experiments/017-gray-logit-refiner/artifacts/refiner-gray-logits-seed42.pt"
SEED = 42
DEVICE = "mps"
N_CROPS = 32
BANK_CROP_PX = 512
RARE_CROP_P = 0.3
GAP_PX = 64
CROP_PX = 256
BATCH = 16
OVERFIT_STEPS = 400
OVERFIT_LR = 3e-3
LAMBDA_DICE = 1.0
VARIANTS = {"017 as run": {}, "+ input norm": {"input_norm": True}, "+ pointwise & skip": {"pointwise_skip": True},
            "+ both": {"input_norm": True, "pointwise_skip": True}}

# Data

In [ ]:
set_seed(SEED)
device = torch.device(DEVICE if DEVICE != "mps" or torch.backends.mps.is_available() else "cpu")
DERIVED = REPO_ROOT / "data" / "derived"
sat = vs.load_satellite(DERIVED / "ua-village-annotated", DERIVED)
vs.ban_holdouts(sat, HOLDOUT_LOCATION, STRIP_LOCATION, STRIP_ROWS, GAP_PX)
sat_train = [s for s in sat if not s.banned.all()]
sampler = vs.SatelliteSampler(sat_train, BANK_CROP_PX, RARE_CROP_P)
bank = build_crop_bank(load_baseline_model(device), sampler, N_CROPS, 8, SEED, device)

In [ ]:
x, z, y = bank_batch(bank, np.arange(BATCH), CROP_PX, np.random.default_rng(SEED), device, as_logits=True)
x.shape, z.shape, y.shape

# Model

In [ ]:
trained = ResidualRefiner(ResidualRefinerConfig())
trained.load_state_dict(torch.load(REPO_ROOT / CHECKPOINT, map_location="cpu"))
trained = trained.to(device).eval()

# Weights

In [ ]:
pd.DataFrame({name: {"norm": p.norm().item(), "abs_mean": p.abs().mean().item(), "numel": p.numel()}
              for name, p in trained.named_parameters()}).T.round(4)

# Input scale

In [ ]:
gray = luma(x)
channels = {"gray": gray[:, 0], **{f"z{c}": z[:, c] for c in range(z.shape[1])}}
pd.DataFrame({k: {"mean": v.mean().item(), "std": v.std().item(), "min": v.min().item(), "max": v.max().item()}
              for k, v in channels.items()}).T.round(3)

In [ ]:
def stem_share(model: ResidualRefiner) -> dict:
    """Norm of the stem's pre-activation coming from the gray channel vs from the logits, on this batch."""
    w, b = model.stem.weight, model.stem.bias
    from_gray = F.conv2d(luma(x), w[:, :1], padding=1)
    from_logits = F.conv2d(z, w[:, 1:], padding=1)
    return {"gray": from_gray.norm().item(), "logits": from_logits.norm().item(), "bias": (b.norm() * np.sqrt(z[:, 0].numel())).item()}

In [ ]:
with torch.no_grad():
    share = stem_share(trained)
share

# Activations

In [ ]:
@torch.no_grad()
def activation_stats(model: ResidualRefiner) -> pd.DataFrame:
    """Per layer: fraction of exactly-zero ReLU outputs, and channels that are zero on the whole batch (dead)."""
    image = luma(x) if model.config.image == "gray" else x
    h = F.relu(model.stem(torch.cat([image, z], 1)))
    rows = {"stem": h}
    for d, conv in zip(model.config.dilations, model.depthwise):
        h = F.relu(conv(h))
        rows[f"dw d={d}"] = h
    return pd.DataFrame({k: {"zero_frac": (v == 0).float().mean().item(),
                             "dead_channels": int(((v != 0).sum((0, 2, 3)) == 0).sum()),
                             "mean": v.mean().item()} for k, v in rows.items()}).T

In [ ]:
activation_stats(trained).round(3)

# Correction size

In [ ]:
@torch.no_grad()
def correction_stats(model: ResidualRefiner) -> dict:
    """Class-relative |ΔZ| against the top-1 minus top-2 logit margin, and how many pixels change class."""
    d = model.delta(x, z)
    d = d - d.mean(1, keepdim=True)
    top2 = z.topk(2, 1).values
    margin = top2[:, 0] - top2[:, 1]
    flipped = (z + d).argmax(1) != z.argmax(1)
    ok = y != 255
    return {"delta_abs_median": d.abs().amax(1).median().item(), "delta_abs_p99": d.abs().amax(1).quantile(.99).item()
            if d.numel() < 2 ** 24 else float("nan"), "margin_median": margin.median().item(),
            "flipped_frac": flipped.float().mean().item(),
            "flip_fixes": (flipped & ok & ((z + d).argmax(1) == y)).sum().item(),
            "flip_breaks": (flipped & ok & (z.argmax(1) == y)).sum().item()}

In [ ]:
correction_stats(trained)

# Overfit one batch

In [ ]:
def overfit(model: torch.nn.Module, steps: int, name: str) -> list[float]:
    """Train on the single fixed batch `(x, z, y)`; returns the CE per step. A healthy model drives it far down."""
    set_seed(SEED)
    model = model.to(device).train()
    opt = torch.optim.AdamW(model.parameters(), lr=OVERFIT_LR)
    history = []
    bar = tqdm(range(steps), desc=name, unit="step")
    for _ in bar:
        scores = model(x, z)
        loss = ce_dice_loss(scores, y, dice_weight=LAMBDA_DICE)
        opt.zero_grad(); loss.backward(); opt.step()
        history.append(cross_entropy(scores.detach(), y).item())
        bar.set_postfix(ce=f"{history[-1]:.4f}")
    return history

In [ ]:
curves = {"identity (LR-ASPP)": [cross_entropy(z, y).item()] * OVERFIT_STEPS}
for name, options in VARIANTS.items():
    set_seed(SEED)
    curves[name] = overfit(ResidualRefiner(ResidualRefinerConfig(**options)), OVERFIT_STEPS, name)

In [ ]:
set_seed(SEED)
curves["013 RefinementCNN (fed logits)"] = overfit(RefinementCNN(RefinerConfig()), OVERFIT_STEPS, "013")

In [ ]:
pd.DataFrame(curves).iloc[[0, 50, 100, 200, OVERFIT_STEPS - 1]].T.round(4)

# Plots

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, h in curves.items():
    ax.plot(h, label=label)
ax.set_xlabel("step on one fixed batch"); ax.set_ylabel("CE"); ax.grid(alpha=.3); ax.legend()
fig.tight_layout()